# Llama-3.2-3B student server (Colab T4)

Serves the student model for the L-DRL experiments so the Mac only sends prompts and scores answers.

**How to use**
1. Connect this notebook to a Colab runtime with a **T4 GPU**.
2. Run all cells.
3. Copy the two lines printed by the last setup cell into the project's `.env`:
   `SMALL_MODEL_REMOTE_URL=...` and `SMALL_MODEL_REMOTE_KEY=...`
4. Leave the final cell running while experiments run; the runtime must stay connected.

Same model weights (`unsloth/Llama-3.2-3B-Instruct`, fp16), same greedy decoding, same loop guard and
time cap as the local backend (`LocalSmallModelClient`). Jobs run one at a time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install fastapi uvicorn

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO = "unsloth/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(REPO)
mdl = AutoModelForCausalLM.from_pretrained(REPO, dtype=torch.float16).to("cuda").eval()
print("loaded", REPO, next(mdl.parameters()).dtype, torch.cuda.get_device_name(0))

Loop guard — an exact copy of `src/lex_drl/loopguard.py`, so the GPU stops looping answers at the same place the Mac would.

In [ ]:
# --- copied verbatim from src/lex_drl/loopguard.py ---
"""Stop a small model that has started repeating itself.

With greedy decoding, Llama-3.2-3B sometimes falls into a loop when the prompt
is long: it writes the same JSON node again and again with only the id
changing ("R10 … §20-873", "R13 … §20-873", …) until it hits the token limit,
and the graph never gets its applications or conclusions.

A loop is detected when one JSON node line, with its id removed, has appeared
``repeats`` times, or when one node type passes ``MAX_PER_TYPE`` lines (the
other failure: inventing §20-923, §20-924, … one after another). The teacher
never writes more than 27 nodes of one type; edges are not capped. The answer is cut just before the
offending line. Because decoding is
greedy, stopping generation there (``LoopGuard``) and cutting a finished answer
there (``truncate_loop``) give the same text, so cached answers can be guarded
after the fact without regenerating them.
"""

import re
from typing import Optional

_ID_FIELD = re.compile(r'"(fid|iid|rid|aid|cid|oid|eid)"\s*:\s*"[A-Z]\d+"')
MIN_LINE = 40          # shorter lines ("],", "{") repeat legitimately
REPEATS = 3
MAX_PER_TYPE = 30


def _norm(line: str) -> str:
    return _ID_FIELD.sub('"id"', line.strip()).rstrip(",")


def loop_start(text: str, repeats: int = REPEATS,
               max_per_type: int = MAX_PER_TYPE) -> Optional[int]:
    """Character offset of the line where a loop is detected, or None.

    Only complete JSON node lines (ones carrying an id field) are counted, so
    repeated prose or brackets never trigger it.
    """
    counts: dict[str, int] = {}
    per_type: dict[str, int] = {}
    pos = 0
    for line in text.splitlines(keepends=True):
        m = _ID_FIELD.search(line)
        if m and line.endswith("\n"):
            kind = m.group(1)
            per_type[kind] = per_type.get(kind, 0) + 1
            if kind != "eid" and per_type[kind] > max_per_type:   # edges are legitimately many
                return pos
            key = _norm(line)
            if len(key) >= MIN_LINE:
                counts[key] = counts.get(key, 0) + 1
                if counts[key] >= repeats:
                    return pos
        pos += len(line)
    return None


def truncate_loop(text: str, repeats: int = REPEATS) -> tuple[str, bool]:
    """(text cut before the looping line, whether a loop was found)."""
    cut = loop_start(text, repeats)
    return (text, False) if cut is None else (text[:cut], True)


class LoopGuard:
    """transformers StoppingCriteria: stop once ``loop_start`` fires.

    Only checks when the newest token contains a newline, so the cost is one
    decode per generated line.
    """

    def __init__(self, tokenizer, prompt_len: int, repeats: int = REPEATS, preamble: str = ""):
        # ``preamble``: answer text already written before this generation (when
        # continuing a repaired answer), so a loop spanning both is still caught.
        self.tok, self.prompt_len, self.repeats = tokenizer, prompt_len, repeats
        self.preamble = preamble
        self.fired = False

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        last = self.tok.decode(input_ids[0, -1:], skip_special_tokens=True)
        if "\n" not in last:
            return False
        text = self.tok.decode(input_ids[0, self.prompt_len:], skip_special_tokens=True,
                               clean_up_tokenization_spaces=False)
        self.fired = loop_start(self.preamble + text, self.repeats) is not None
        return self.fired


__all__ = ["loop_start", "truncate_loop", "LoopGuard", "REPEATS", "MAX_PER_TYPE"]

In [ ]:
import threading, time, uuid, queue
from transformers import MaxTimeCriteria, StoppingCriteriaList

def _messages(system, user):
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def run_job(job):
    """Mirrors LocalSmallModelClient.generate / continue_from."""
    enc = tok.apply_chat_template(_messages(job["system"], job["user"]), add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    ids = enc["input_ids"]
    prefix = job.get("prefix", "") if job["kind"] == "continue" else ""
    max_new = int(job.get("max_tokens", 4096))
    if prefix:
        pre = tok(prefix, add_special_tokens=False, return_tensors="pt")["input_ids"]
        ids = torch.cat([ids, pre], dim=-1)
        max_new = max(768, max_new - pre.shape[-1])
    ids = ids.to("cuda")
    in_len = ids.shape[-1]
    crit = StoppingCriteriaList()
    if job.get("loop_guard"):
        crit.append(LoopGuard(tok, in_len, preamble=prefix))
    if job.get("max_seconds"):
        crit.append(MaxTimeCriteria(max_time=float(job["max_seconds"])))
    with torch.no_grad():
        out = mdl.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
                           do_sample=False, pad_token_id=tok.eos_token_id,
                           stopping_criteria=crit if len(crit) else None)
    gen = out[0][in_len:]
    text = tok.decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    torch.cuda.empty_cache()
    return {"text": text, "input_tokens": int(in_len), "output_tokens": int(gen.shape[-1])}

JOBS, Q = {}, queue.Queue()

def worker():
    while True:
        jid = Q.get()
        job = JOBS[jid]
        job["status"] = "running"; t0 = time.time()
        try:
            job.update(run_job(job["req"]), status="done")
        except Exception as e:
            job.update(status="error", error=repr(e)[:500])
        job["seconds"] = round(time.time() - t0, 1)
        print(f"[{time.strftime('%H:%M:%S')}] {job['req']['kind']:8} {job['status']:5} "
              f"{job.get('output_tokens', 0):5} tokens  {job['seconds']:6.1f}s", flush=True)

threading.Thread(target=worker, daemon=True).start()

In [ ]:
import secrets, uvicorn
from fastapi import FastAPI, Header, HTTPException

API_KEY = secrets.token_urlsafe(24)
app = FastAPI()

def _auth(key):
    if key != API_KEY:
        raise HTTPException(status_code=401, detail="bad key")

@app.post("/submit")
def submit(req: dict, x_api_key: str = Header(None)):
    _auth(x_api_key)
    jid = uuid.uuid4().hex
    JOBS[jid] = {"status": "queued", "req": req}
    Q.put(jid)
    return {"id": jid, "queued": Q.qsize()}

@app.get("/result/{jid}")
def result(jid: str, x_api_key: str = Header(None)):
    _auth(x_api_key)
    job = JOBS.get(jid)
    if job is None:
        raise HTTPException(status_code=404, detail="unknown job")
    out = {k: v for k, v in job.items() if k != "req"}
    if job["status"] in ("done", "error"):
        JOBS.pop(jid, None)            # results are fetched once
    return out

@app.get("/health")
def health():
    return {"ok": True, "queued": Q.qsize(), "gpu": torch.cuda.get_device_name(0)}

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("server up")

In [ ]:
# Public HTTPS tunnel (Cloudflare quick tunnel, no account needed)
import subprocess, re
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep draining its log
time.sleep(5)
print("Add these two lines to the project's .env:\n")
print(f"SMALL_MODEL_REMOTE_URL={URL}")
print(f"SMALL_MODEL_REMOTE_KEY={API_KEY}")

Keep this cell running while experiments run. It prints one line per finished job and keeps the runtime busy.

In [ ]:
while True:
    time.sleep(300)
    print(f"[{time.strftime('%H:%M:%S')}] alive — queued {Q.qsize()}, "
          f"GPU mem {torch.cuda.memory_allocated() / 2**30:.1f} GB", flush=True)